# COMET offline — 5 cấu hình weight

- không internet
- dùng checkpoint + XLM-R cache đã attach
- 1688: COMET-da; Tiki: COMET-QE


In [ ]:
import os, sys, subprocess, zipfile, json, time
from pathlib import Path

# cài wheel offline trước khi import comet
WHEELS = Path('/kaggle/input/tiki-comet-wheels')
if WHEELS.exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index',
                    '--find-links', str(WHEELS), '--no-deps',
                    'unbabel-comet==2.2.7', 'pytorch-lightning==2.6.6',
                    'torchmetrics==1.9.0', 'lightning-utilities==0.15.3',
                    'entmax==1.3', 'sacrebleu==2.6.0',
                    'jsonargparse', 'pyyaml'], check=False)
print('wheels done')


In [ ]:
# tìm checkpoint + HF cache; đặt biến môi trường offline trước khi import HF
FILES = Path('/kaggle/input/comet-offline-files')
WORK = Path('/kaggle/working/comet'); WORK.mkdir(parents=True, exist_ok=True)
if not (FILES / 'wmt22-da.ckpt').exists():
    print('input dirs', [p.name for p in Path('/kaggle/input').iterdir()])
    raise SystemExit('không thấy checkpoint COMET trong /kaggle/input/comet-offline-files')
for name in ['wmt22-da','wmt20-qe']:
    d = WORK / f'checkpoints/{name}/checkpoints'; d.mkdir(parents=True, exist_ok=True)
    ck = d / 'model.ckpt'
    if not ck.exists(): ck.symlink_to(FILES / f'{name}.ckpt')
    hp = WORK / f'checkpoints/{name}/hparams.yaml'
    if not hp.exists(): hp.symlink_to(FILES / f'{name}.hparams.yaml')
if not (WORK / 'hf').exists():
    with zipfile.ZipFile(FILES / 'xlm-roberta-large.zip') as zf:
        zf.extractall(WORK)
COMET_BASE = WORK
os.environ['HF_HOME'] = str(WORK / 'hf')
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['HF_HUB_OFFLINE'] = '1'
print('comet base', COMET_BASE)
print('checkpoint MB', round((COMET_BASE/'checkpoints/wmt22-da/checkpoints/model.ckpt').stat().st_size/1e6,1))


In [ ]:
# patch COMET XLM-R cho transformers 5.x (add_pooling_layer=False trả 2 tuple)
import comet
p = Path(comet.__file__).parent / 'encoders/xlmr.py'
s = p.read_text()
old = '''        last_hidden_states, _, all_layers = self.model(\n            input_ids=input_ids,\n            attention_mask=attention_mask,\n            output_hidden_states=True,\n            return_dict=False,\n        )'''
new = '''        outputs = self.model(\n            input_ids=input_ids,\n            attention_mask=attention_mask,\n            output_hidden_states=True,\n            return_dict=False,\n        )\n        if len(outputs) == 3:\n            last_hidden_states, _, all_layers = outputs\n        else:\n            last_hidden_states, all_layers = outputs'''
if old in s:
    p.write_text(s.replace(old, new)); print('patched')
else:
    print('patch already applied or source differs')


In [ ]:
import torch, pandas as pd, numpy as np
from comet import load_from_checkpoint
print('gpu', torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'GB')
t0=time.time()
da = load_from_checkpoint(str(ROOT/'checkpoints/wmt22-da/checkpoints/model.ckpt'), local_files_only=True)
qe = load_from_checkpoint(str(ROOT/'checkpoints/wmt20-qe/checkpoints/model.ckpt'), local_files_only=True)
print('loaded in', round(time.time()-t0,1), 's')


In [ ]:
HYP = Path('/kaggle/input/tiki-weight-hyps')
OUT = Path('/kaggle/working/scores'); OUT.mkdir(parents=True, exist_ok=True)
metrics = {}
for path in sorted(HYP.glob('hyp_*.parquet')):
    df = pd.read_parquet(path).drop_duplicates('eval_id')
    config = str(df['config'].iloc[0]); seed = int(df['seed'].iloc[0])
    for dataset in ['1688','tiki']:
        part = df[df['dataset']==dataset].copy()
        part = part[part['hypothesis'].astype(str).str.strip().ne('')]
        if dataset=='1688':
            part = part[part['reference'].astype(str).str.strip().ne('')]
            data=[{'src':str(s),'mt':str(m),'ref':str(r)} for s,m,r in zip(part.source,part.hypothesis,part.reference)]
            out=da.predict(data, batch_size=8, gpus=1)
            import sacrebleu
            bleu=sacrebleu.corpus_bleu(part.hypothesis.tolist(),[part.reference.tolist()],tokenize='zh').score
            chrf=sacrebleu.corpus_chrf(part.hypothesis.tolist(),[part.reference.tolist()]).score/100
            extra={'bleu':round(float(bleu),4),'chrf':round(float(chrf),4)}
        else:
            data=[{'src':str(s),'mt':str(m)} for s,m in zip(part.source,part.hypothesis)]
            out=qe.predict(data, batch_size=8, gpus=1)
            extra={}
        part['score']=np.asarray(out.scores,dtype=float)
        part['config']=config; part['seed']=seed; part['dataset']=dataset
        part[['eval_id','pair_type','score','config','seed','dataset']].to_parquet(OUT/f'scores_{config}_seed{seed}_{dataset}.parquet',index=False)
        key=f'{config}::{seed}::{dataset}'
        metrics[key]={'system_score':round(float(out.system_score),4),'n':int(len(part)),**extra}
        print(key, metrics[key], flush=True)
(OUT/'comet_scores.json').write_text(json.dumps(metrics,ensure_ascii=False,indent=2),encoding='utf-8')
print(json.dumps(metrics,ensure_ascii=False,indent=2))
